# 08 — NumPy: The Engine Under Every ML Framework

**Difficulty:** ⭐⭐ | **Estimated Time:** 3 hours  
**Week 3 — Statistics for ML & Python Data Stack**

---

> *"NumPy is to Python what a calculator is to arithmetic — it doesn't change what you're doing, just how fast you do it."*

---

## Learning Objectives
By the end of this notebook you will be able to:
- Explain why NumPy arrays are dramatically faster than Python lists for numerical computation
- Create arrays using all common constructors (`np.array`, `np.zeros`, `np.linspace`, etc.)
- Index and slice 1D and 2D arrays including boolean indexing
- Apply vectorized operations and universal functions (ufuncs) instead of loops
- Use axis-based aggregations (sum, mean, std) on matrices
- Understand and apply broadcasting to normalize data in one line
- Generate synthetic datasets using the NumPy random module
- Understand that every pandas DataFrame, every sklearn model, every neural network uses NumPy arrays internally

## 1. Why Does This Matter for Machine Learning?

**NumPy is the foundation of the entire Python ML ecosystem.** When you:

- Load data with `pandas.read_csv()` — the underlying storage is NumPy arrays
- Train an `sklearn` model — internally it converts your DataFrame to NumPy
- Use `tensorflow` or `pytorch` — tensors are conceptually NumPy arrays on GPU
- Compute mean, std, dot products anywhere — NumPy is doing the math

You can build ML models without knowing NumPy — just like you can drive a car without knowing how the engine works. But when something goes wrong (shape mismatch errors, unexpected NaNs, slow preprocessing), knowing NumPy lets you debug in minutes instead of hours.

**Concrete ML uses of NumPy you'll encounter:**
| Operation | NumPy function |
|-----------|---------------|
| Feature matrix (X) | `np.array` with shape (n_samples, n_features) |
| Normalize features | Broadcasting: `(X - X.mean(axis=0)) / X.std(axis=0)` |
| Log-transform skewed prices | `np.log(prices)` |
| Train/test split | `np.random.shuffle`, boolean indexing |
| Dot product (linear regression) | `np.dot(X, weights)` |
| Distance matrix (KNN) | `np.linalg.norm` |
| Generate synthetic data | `np.random.normal`, `np.random.randint` |

## 2. The Big Analogy: Shipping Container vs. Individual Packages

Imagine you need to ship 1,000 boxes from a warehouse.

**Python list approach:** A courier picks up each box individually, drives to the destination, comes back, picks up the next box. 1,000 trips. Huge overhead per box.

**NumPy array approach:** Load all 1,000 boxes into a single shipping container. One trip. The container is handled as a unit.

```
Python list:   [box1] → [box2] → [box3] → ... → [box1000]
               (1000 separate Python operations, each with overhead)

NumPy array:   [box1 | box2 | box3 | ... | box1000]
               (1 C-level operation on contiguous memory block)
```

**Technical reason:** Python lists store **references** to Python objects (each object has a type, reference count, memory address). NumPy arrays store **raw numbers in contiguous C memory** — no per-element overhead. The operations are compiled C code, not interpreted Python.

### Memory layout:
```
Python list:   [ptr→300000] [ptr→250000] [ptr→410000] [ptr→...]  ← pointers scattered in memory
                                                                     each element is a Python object

NumPy array:   [300000.0][250000.0][410000.0][...]                ← numbers packed tight in memory
               ^—— all float64 values, 8 bytes each, sequential ——^
```

In [ ]:
# ============================================================
# SETUP: Import NumPy and set up for demonstrations
# ============================================================
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import time  # for manual timing comparisons

# Set global random seed for reproducibility
np.random.seed(42)

# Configure plot style
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 12
sns.set_theme(style='whitegrid')

print(f"NumPy version: {np.__version__}")

# Check the underlying C library NumPy is using
np.show_config()  # shows BLAS/LAPACK info — these are highly optimized C/Fortran libraries

In [ ]:
# ============================================================
# PERFORMANCE COMPARISON: Python list vs NumPy array
# Summing 1 million house prices
# ============================================================
N = 1_000_000  # 1 million houses — a realistic large dataset size

# Create data as both Python list and NumPy array
prices_list  = list(range(100000, 100000 + N))   # Python list
prices_array = np.arange(100000, 100000 + N,     # NumPy array (same values)
                          dtype=np.float64)

# --- Time the Python list sum ---
start = time.perf_counter()          # high-resolution timer
result_list = sum(prices_list)       # Python built-in sum: iterates one by one
time_list = time.perf_counter() - start

# --- Time the NumPy sum ---
start = time.perf_counter()
result_numpy = np.sum(prices_array)  # NumPy sum: compiled C code on contiguous memory
time_numpy = time.perf_counter() - start

print(f"Summing {N:,} house prices:")
print(f"  Python list sum:  {time_list*1000:.2f} ms  (result: {result_list:,.0f})")
print(f"  NumPy np.sum():   {time_numpy*1000:.2f} ms  (result: {result_numpy:,.0f})")
print(f"  NumPy is approx {time_list/time_numpy:.0f}x faster!")

print()
print("Memory usage comparison:")
import sys
# Python list: each element is a full Python int object (~28 bytes)
list_mem = sys.getsizeof(prices_list) + sum(sys.getsizeof(x) for x in prices_list[:10]) * N // 10
# NumPy array: each element is 8 bytes (float64), no Python object overhead
numpy_mem = prices_array.nbytes  # nbytes = total bytes used by array data

print(f"  Python list (estimated): ~{list_mem / 1e6:.0f} MB")
print(f"  NumPy array:             {numpy_mem / 1e6:.1f} MB (8 bytes × {N:,} elements)")
print(f"  NumPy uses ~{list_mem/numpy_mem:.0f}x less memory!")

## 3. Array Creation — Your Toolkit

NumPy provides many ways to create arrays. Here's the complete toolkit you'll use in ML:

| Constructor | What it does | When to use |
|------------|--------------|-------------|
| `np.array([...])` | Convert Python list to array | Loading data, quick tests |
| `np.zeros(n)` | Array of zeros | Initialize weights in neural nets |
| `np.ones(n)` | Array of ones | Bias terms, masks |
| `np.full(n, val)` | Array filled with `val` | Initialize to specific value |
| `np.arange(start, stop, step)` | Evenly spaced like Python `range` | Integer sequences |
| `np.linspace(start, stop, n)` | n evenly spaced floats | Plot x-axes, parameter grids |
| `np.eye(n)` | n×n identity matrix | Linear algebra, matrix operations |
| `np.empty(shape)` | Uninitialized array (random garbage) | When you'll fill every element |
| `np.random.normal(...)` | Random normal distribution | Synthetic data, noise |
| `np.random.randint(...)` | Random integers | Discrete features, sampling |

In [ ]:
# ============================================================
# ARRAY CREATION: All the constructors with house price context
# ============================================================

# --- From a Python list: specific house prices you know ---
known_prices = np.array([320000, 450000, 285000, 610000, 390000])
print(f"From list:           {known_prices}")
print(f"  dtype={known_prices.dtype}, shape={known_prices.shape}, ndim={known_prices.ndim}")

# --- Zeros: placeholder for predictions before we compute them ---
predictions = np.zeros(5)  # 5 empty prediction slots
print(f"\nZeros (predictions): {predictions}")

# --- Ones: all houses start with a weight of 1 (equal importance) ---
weights = np.ones(5)
print(f"Ones (weights):      {weights}")

# --- Full: initialize all predictions to the dataset mean price ---
mean_price = 400000
baseline = np.full(5, mean_price)  # simplest model: always predict average
print(f"Full (mean baseline): {baseline}")

# --- Arange: house IDs 1 through 10 ---
house_ids = np.arange(1, 11, 1)   # start=1, stop=11 (exclusive), step=1
print(f"\nArange (IDs 1–10):   {house_ids}")

# Price bins from $100K to $1M in $100K steps
price_bins = np.arange(100_000, 1_100_000, 100_000)
print(f"Price bins:          {price_bins}")

# --- Linspace: 6 evenly spaced prices between $200K and $800K ---
# Unlike arange, linspace gives you exactly n points
# (arange gives you values BY step, linspace gives you exactly n values)
price_range = np.linspace(200_000, 800_000, 6)
print(f"\nLinspace (6 prices): {price_range}")

# --- Eye (identity matrix): 4×4 — used in matrix operations ---
I = np.eye(4)
print(f"\nIdentity matrix 4×4:")
print(I)
print("(Used in linear regression: (X'X)^{-1} involves identity-like operations)")

## 4. Array Attributes — Understanding What You Have

Before operating on any array, know these four attributes:

| Attribute | What it tells you | Example |
|-----------|------------------|--------|
| `.shape` | Dimensions as a tuple | `(500, 5)` = 500 rows, 5 columns |
| `.dtype` | Data type of elements | `float64`, `int32`, `bool` |
| `.ndim` | Number of dimensions | 1 (vector), 2 (matrix), 3 (tensor) |
| `.size` | Total number of elements | `shape[0] * shape[1] * ...` |

**In ML:** `X.shape` tells you (n_samples, n_features). You'll check this constantly when debugging shape mismatch errors.

In [ ]:
# ============================================================
# ARRAY ATTRIBUTES: Exploring the feature matrix
# ============================================================
np.random.seed(42)

# Create a 2D feature matrix: 10 houses × 4 features
# Columns: [size_sqft, bedrooms, bathrooms, age_years]
X = np.array([
    [1800, 3, 2, 10],
    [2500, 4, 3, 5 ],
    [1200, 2, 1, 40],
    [3100, 5, 3, 2 ],
    [1600, 3, 2, 25],
    [2200, 4, 2, 8 ],
    [900,  2, 1, 55],
    [4000, 5, 4, 3 ],
    [1400, 3, 2, 30],
    [2800, 4, 3, 7 ]
], dtype=np.float64)  # float64 for ML (sklearn requires float, not int)

# Corresponding prices
y = np.array([320000, 450000, 190000, 560000, 280000,
              380000, 140000, 720000, 240000, 490000], dtype=np.float64)

print("=== Feature Matrix X (10 houses, 4 features) ===")
print("Columns: [size_sqft, bedrooms, bathrooms, age_years]")
print(X)

print(f"\nX.shape  = {X.shape}     ← (n_samples=10, n_features=4)")
print(f"X.dtype  = {X.dtype}    ← 64-bit float (standard for ML)")
print(f"X.ndim   = {X.ndim}           ← 2D array (matrix)")
print(f"X.size   = {X.size}          ← total elements = 10 × 4 = 40")

print(f"\ny.shape  = {y.shape}      ← 1D array of 10 prices")
print(f"y[:5]    = {y[:5]}  ← first 5 prices")

# Reshape: sometimes sklearn needs y as a column vector
y_col = y.reshape(-1, 1)   # -1 means "figure out this dimension automatically"
print(f"\ny.reshape(-1,1).shape = {y_col.shape}  ← (n_samples, 1) column vector")

## 5. Indexing and Slicing

Think of indexing like addressing houses on a street (1D) or rooms in a building floor-plan (2D).

### 1D Indexing
```
prices = [300K, 250K, 410K, 180K, 520K]
index:     0     1     2     3     4
neg idx:  -5    -4    -3    -2    -1    (count from the end)
```

### 2D Indexing
```
X[row, col]  — specific cell
X[:, col]    — entire column (all rows, one column)
X[row, :]    — entire row (one house, all features)
X[2:5, :]    — rows 2,3,4 (houses 2–4), all features
```

### Boolean Indexing — The Powerful One
Instead of specifying positions, you give NumPy a mask of True/False values:
```python
prices[prices > 400000]  # only expensive houses
X[y > 400000, :]         # rows of X where price > 400K
```
This is how you filter data in ML preprocessing — no loops needed!

In [ ]:
# ============================================================
# INDEXING AND SLICING: 1D and 2D with house data
# ============================================================
print("=== 1D Indexing: House Prices ===")
prices_demo = np.array([320000, 450000, 190000, 560000, 280000,
                        380000, 140000, 720000, 240000, 490000])

print(f"prices_demo[0]      = {prices_demo[0]:>10,}  ← first house")
print(f"prices_demo[-1]     = {prices_demo[-1]:>10,}  ← last house (negative indexing)")
print(f"prices_demo[2:5]    = {prices_demo[2:5]}  ← houses at index 2,3,4")
print(f"prices_demo[::2]    = {prices_demo[::2]}  ← every other house (step=2)")
print(f"prices_demo[::-1]   = {prices_demo[::-1]}  ← reversed")

print("\n=== 2D Indexing: Feature Matrix ===")
print("X columns: [size_sqft, bedrooms, bathrooms, age_years]")
print()

print(f"X[0]          = {X[0]}    ← first house (all features)")
print(f"X[0, 0]       = {X[0, 0]:.0f}         ← first house, first feature (size)")
print(f"X[0, 2]       = {X[0, 2]:.0f}            ← first house, bathrooms (col index 2)")
print(f"X[3, :]       = {X[3, :]}  ← house 4, all features")

# Select all values of a specific feature (column) — very common in ML
all_sizes = X[:, 0]   # column 0 = size_sqft, all rows
print(f"X[:, 0]       = {all_sizes.astype(int)}  ← ALL house sizes")

all_bedrooms = X[:, 1]  # column 1 = bedrooms
print(f"X[:, 1]       = {all_bedrooms.astype(int)}  ← ALL bedroom counts")

# Slice a sub-matrix: first 3 houses, first 2 features
sub_matrix = X[:3, :2]
print(f"\nX[:3, :2] (first 3 houses, size & bedrooms):")
print(sub_matrix)

print("\n=== Boolean Indexing: Filter by Price ===")
# Create a boolean mask: True where price > 400,000
expensive_mask = y > 400000  # array of True/False
print(f"y > 400000 mask: {expensive_mask}")
print(f"Expensive prices: {y[expensive_mask]}")  # use mask to select values
print(f"Number of houses > $400K: {expensive_mask.sum()}")

# Use same mask to get the features of expensive houses
expensive_houses = X[expensive_mask, :]  # rows where mask is True
print(f"\nFeatures of expensive houses (n={expensive_houses.shape[0]}):")
print(expensive_houses)

In [ ]:
# ============================================================
# MORE BOOLEAN INDEXING: compound conditions
# ============================================================

# Find houses that are both large AND expensive
sizes = X[:, 0]   # size_sqft
large_and_expensive = (sizes > 2000) & (y > 400000)  # & is element-wise AND
print("Houses with size > 2000 sqft AND price > $400K:")
print(f"  Indices: {np.where(large_and_expensive)[0]}")
print(f"  Sizes:   {sizes[large_and_expensive].astype(int)}")
print(f"  Prices:  {y[large_and_expensive]}")

# Find cheap OR small houses
cheap_or_small = (y < 250000) | (sizes < 1400)  # | is element-wise OR
print(f"\nHouses with price < $250K OR size < 1400 sqft: {cheap_or_small.sum()} found")

# NOT operator: houses that are NOT new (age > 5 years)
ages = X[:, 3]    # age_years column
not_new = ~(ages <= 5)  # ~ is element-wise NOT
print(f"Houses older than 5 years: {not_new.sum()} out of {len(ages)}")

# np.where: vectorized if/else — use value_if_true where condition, else value_if_false
price_category = np.where(y > 400000, 'expensive', 'affordable')
print(f"\nPrice categories: {price_category}")

# Fancy indexing: select specific rows by index
selected_rows = X[[0, 3, 7], :]  # select houses 0, 3, and 7
print(f"\nFancy indexing X[[0, 3, 7], :] — 3 specific houses:")
print(selected_rows)

## 6. Vectorized Operations — Replacing Loops

The key shift in thinking when using NumPy:

**Loop mindset:** "I'll go through each house and apply the calculation."

**NumPy mindset:** "I'll apply the calculation to the whole array at once."

```python
# SLOW: Python loop
adjusted = []
for price in prices:
    adjusted.append(price * 1.1)

# FAST: NumPy vectorization
adjusted = prices * 1.1  # one operation on the whole array
```

Both give the same result, but the NumPy version is:
- 10–100x faster (compiled C code)
- One line instead of three
- No risk of loop-related bugs (off-by-one errors, etc.)

### Universal Functions (ufuncs)

NumPy provides functions that automatically apply element-by-element:
- `np.sqrt(arr)` — square root of every element
- `np.log(arr)` — natural log of every element (crucial for log-transforming skewed prices)
- `np.exp(arr)` — e^x for every element
- `np.abs(arr)` — absolute value
- `np.square(arr)` — square every element
- `np.round(arr, decimals)` — round every element

In [ ]:
# ============================================================
# VECTORIZED OPERATIONS: Arithmetic and ufuncs
# ============================================================
print("=== Arithmetic Operations (operate on entire array at once) ===")
prices_v = np.array([320000., 450000., 190000., 560000., 280000.])

# Scalar operations: applied to every element simultaneously
print(f"Original prices:       {prices_v}")
print(f"+ $50,000 renovation:  {prices_v + 50000}")
print(f"10% price increase:    {prices_v * 1.10}")
print(f"Price in $K:           {prices_v / 1000}")

# Array + array: element-wise (both must be same shape)
renovation_cost = np.array([20000., 0., 50000., 10000., 30000.])
value_after_reno = prices_v + renovation_cost  # element-wise addition
print(f"\nAfter renovation:      {value_after_reno}")

print("\n=== Universal Functions (ufuncs) ===")

# Log transformation: house prices are right-skewed
# log makes the distribution more normal → better for linear models
log_prices = np.log(prices_v)  # natural log
print(f"Original prices:  {prices_v}")
print(f"Log-transformed:  {np.round(log_prices, 2)}")

# Back-transform: use exp to reverse the log
back_transformed = np.exp(log_prices)
print(f"Back-transformed: {np.round(back_transformed, 2)}  ← should match original")

# Square root: sometimes used to reduce skewness (less aggressive than log)
sqrt_prices = np.sqrt(prices_v)
print(f"\nSqrt of prices:   {np.round(sqrt_prices, 2)}")

# Comparison: return boolean arrays
above_300k = prices_v > 300000
print(f"\nAbove $300K mask: {above_300k}")

# Math on 2D matrix: applies to every element
print(f"\nX * 0 (zero out the matrix):")
print(X[:3] * 0)  # multiply first 3 rows by 0

In [ ]:
# ============================================================
# AGGREGATION FUNCTIONS with axis parameter
# axis=0: collapse rows (compute per-column stats)
# axis=1: collapse columns (compute per-row stats)
# ============================================================
print("Feature Matrix X (10 houses × 4 features):")
print("Columns: size_sqft | bedrooms | bathrooms | age_years")
print(X)

print("\n=== Global aggregations (across all elements) ===")
print(f"np.sum(X)    = {np.sum(X):,.1f}   ← sum of ALL elements in entire matrix")
print(f"np.mean(X)   = {np.mean(X):,.2f}  ← average of all elements (meaningless here — mixed units!)")

print("\n=== axis=0: aggregate across rows (gives one value PER COLUMN) ===")
print("Think: 'collapse all 10 houses into one summary row'")
col_means = np.mean(X, axis=0)  # mean of each feature across all houses
col_stds  = np.std(X, axis=0)   # std of each feature
col_mins  = np.min(X, axis=0)   # min value per feature
col_maxs  = np.max(X, axis=0)   # max value per feature

print(f"  Column means (per feature): {col_means}")
print(f"  Column stds:                {col_stds.round(1)}")
print(f"  Column mins:                {col_mins}")
print(f"  Column maxs:                {col_maxs}")
print("  Each result has shape:", col_means.shape, "(4 values, one per feature)")

print("\n=== axis=1: aggregate across columns (gives one value PER ROW) ===")
print("Think: 'for each house, combine all features into one number'")
row_sums  = np.sum(X, axis=1)   # sum of all feature values for each house
row_means = np.mean(X, axis=1)  # mean of all feature values for each house

print(f"  Row sums (per house):  {row_sums.round(1)}")
print(f"  Row means (per house): {row_means.round(1)}")
print("  Each result has shape:", row_sums.shape, "(10 values, one per house)")

print("\n=== Other useful aggregations ===")
print(f"np.argmax(y) = {np.argmax(y)}  ← INDEX of the most expensive house (price = {y[np.argmax(y)]:,.0f})")
print(f"np.argmin(y) = {np.argmin(y)}  ← INDEX of the cheapest house (price = {y[np.argmin(y)]:,.0f})")
print(f"np.median(y) = {np.median(y):,.0f}  ← median price")
print(f"np.percentile(y, 75) = {np.percentile(y, 75):,.0f}  ← 75th percentile")

## 7. Broadcasting — NumPy's Superpower

Broadcasting is the most powerful (and initially most confusing) NumPy concept. Once you understand it, you'll use it constantly.

**Definition:** Broadcasting allows NumPy to perform operations on arrays of different shapes, by *virtually* expanding the smaller array to match the larger one — without actually copying data.

### The Classic ML Example: Feature Normalization

You have X with shape (10, 4) — 10 houses, 4 features. You want to subtract the mean of each feature from all houses.

```
X.shape        = (10, 4)   ← 10 houses, 4 features
col_means.shape = (4,)     ← 1 mean per feature

X - col_means  → NumPy broadcasts col_means to shape (10, 4)
               → subtracts the CORRECT mean from the CORRECT feature column
               → no loop needed!
```

Visually:
```
        size    beds  baths  age           mean_size mean_beds mean_baths mean_age
house0 [1800    3     2      10  ]         [2250      3.4   2.2    18.5]
house1 [2500    4     3      5   ]         [2250      3.4   2.2    18.5]  ← virtually copied
house2 [1200    2     1      40  ]  minus  [2250      3.4   2.2    18.5]  ← 10 times
...                                        ...

Result: each feature has its own mean subtracted, column by column
```

### Broadcasting Rules
Two arrays are compatible for broadcasting if, for each dimension (from the right), either:
1. They are equal, OR
2. One of them is 1

In [ ]:
# ============================================================
# BROADCASTING: Feature normalization in one line
# ============================================================
print("=== Feature Matrix Before Normalization ===")
print("Columns: [size_sqft, bedrooms, bathrooms, age_years]")
print(X)
print(f"X.shape = {X.shape}")

print("\n=== Step 1: Compute column means and stds ===")
mean_per_feature = X.mean(axis=0)  # shape (4,) — one mean per feature
std_per_feature  = X.std(axis=0)   # shape (4,) — one std per feature

print(f"col_means.shape = {mean_per_feature.shape}")
print(f"col_means = {mean_per_feature.round(2)}")
print(f"col_stds  = {std_per_feature.round(2)}")

print("\n=== Step 2: Standardize via broadcasting (Z-score normalization) ===")
# Standard scaling: subtract mean, divide by std
# X.shape = (10, 4), mean_per_feature.shape = (4,)
# NumPy broadcasts (4,) to (10, 4) automatically
X_standardized = (X - mean_per_feature) / std_per_feature

print(f"X_standardized.shape = {X_standardized.shape}  ← same as X")
print("\nStandardized matrix (each column now has mean≈0, std≈1):")
print(X_standardized.round(3))

# Verify the normalization worked correctly
print("\n=== Verification ===")
print(f"New column means (should be ≈0): {X_standardized.mean(axis=0).round(10)}")
print(f"New column stds  (should be ≈1): {X_standardized.std(axis=0).round(4)}")

print("\n=== Min-Max Normalization: scale to [0, 1] range ===")
# Alternative normalization: (x - min) / (max - min)
X_min  = X.min(axis=0)   # shape (4,)
X_max  = X.max(axis=0)   # shape (4,)
X_normalized = (X - X_min) / (X_max - X_min)  # broadcasting again

print("Min-max normalized matrix (each column in [0, 1]):")
print(X_normalized.round(3))
print(f"\nMin of each column: {X_normalized.min(axis=0)}  ← all zeros ✓")
print(f"Max of each column: {X_normalized.max(axis=0)}  ← all ones ✓")

In [ ]:
# ============================================================
# BROADCASTING VISUALIZATION: see how shapes interact
# ============================================================
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

feature_names = ['size_sqft', 'bedrooms', 'bathrooms', 'age_years']
house_ids_plot = [f'H{i}' for i in range(10)]

# Plot 1: Original feature matrix as heatmap
im1 = axes[0].imshow(X, cmap='YlOrRd', aspect='auto')
axes[0].set_xticks(range(4))
axes[0].set_xticklabels(feature_names, rotation=30, ha='right')
axes[0].set_yticks(range(10))
axes[0].set_yticklabels(house_ids_plot)
axes[0].set_title('Original X\n(different scales!)')
plt.colorbar(im1, ax=axes[0], fraction=0.046, pad=0.04)

# Annotate with values
for i in range(10):
    for j in range(4):
        axes[0].text(j, i, f'{X[i,j]:.0f}', ha='center', va='center', fontsize=7)

# Plot 2: After Z-score standardization
im2 = axes[1].imshow(X_standardized, cmap='coolwarm', aspect='auto', vmin=-2.5, vmax=2.5)
axes[1].set_xticks(range(4))
axes[1].set_xticklabels(feature_names, rotation=30, ha='right')
axes[1].set_yticks(range(10))
axes[1].set_yticklabels(house_ids_plot)
axes[1].set_title('Z-score Standardized\n(mean=0, std=1 per column)')
plt.colorbar(im2, ax=axes[1], fraction=0.046, pad=0.04)

for i in range(10):
    for j in range(4):
        axes[1].text(j, i, f'{X_standardized[i,j]:.2f}', ha='center', va='center', fontsize=7)

# Plot 3: Column-by-column before/after comparison
x_pos = np.arange(4)
width = 0.35
axes[2].bar(x_pos - width/2, X.std(axis=0), width, label='Original std', color='coral', alpha=0.8)
axes[2].bar(x_pos + width/2, X_standardized.std(axis=0), width,
            label='After standardization', color='steelblue', alpha=0.8)
axes[2].set_xticks(x_pos)
axes[2].set_xticklabels(feature_names, rotation=20, ha='right')
axes[2].set_ylabel('Standard Deviation')
axes[2].set_title('Std before vs after normalization\n(all become 1.0 after)')
axes[2].legend()
axes[2].axhline(1.0, color='green', linestyle='--', alpha=0.5, label='Target std=1')

plt.suptitle('Broadcasting in Action: Normalize Features Without Loops',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

print("Why normalization matters for ML:")
print("  size_sqft std = 900+ (huge range)")
print("  bedrooms std  = ~1 (tiny range)")
print("  Without normalization, size_sqft would DOMINATE gradient descent")
print("  After normalization, all features contribute equally to the model")

## 8. The Random Module — Generating Synthetic Data

The NumPy random module is how you generate synthetic datasets, initialize neural network weights, create train/test splits, and add noise to data.

**Two APIs:**
1. **Legacy API:** `np.random.normal(...)`, `np.random.seed(42)` — older, still widely used
2. **Modern API (recommended):** `rng = np.random.default_rng(42)` then `rng.normal(...)` — better for reproducibility in complex code

**Key distributions for ML:**
| Function | What it generates | Use in ML |
|----------|------------------|-----------|
| `np.random.normal(μ, σ, n)` | Bell curve values | Synthetic prices, noise |
| `np.random.uniform(low, high, n)` | Uniform between low–high | Ages, distances |
| `np.random.randint(low, high, n)` | Random integers | Bedrooms, ratings |
| `np.random.choice(arr, n)` | Sample from array | Bootstrapping, sampling |
| `np.random.shuffle(arr)` | Shuffle in-place | Randomize dataset order |
| `np.random.permutation(n)` | Shuffled indices | Train/test split |
| `np.random.binomial(n, p, size)` | Success/fail events | Binary labels |
| `np.random.exponential(scale, n)` | Skewed distribution | Wait times, distances |

In [ ]:
# ============================================================
# THE RANDOM MODULE: Building a synthetic house price dataset
# ============================================================

# Modern API: preferred for new code
rng = np.random.default_rng(seed=42)  # create a random number generator with fixed seed

N_HOUSES = 1000  # dataset size

# --- Continuous features: normal distribution ---
# np.random.normal(mean, std, size) — generates bell-curve values
house_size = rng.normal(loc=2000, scale=500, size=N_HOUSES)   # avg 2000 sqft, std 500
house_size = np.clip(house_size, 500, 6000)                   # no impossibly small/large values

# --- Discrete features: random integers ---
# np.random.randint(low, high, size) — includes low, excludes high
bedrooms  = rng.integers(low=1, high=7, size=N_HOUSES)   # 1–6 bedrooms
bathrooms = rng.integers(low=1, high=5, size=N_HOUSES)   # 1–4 bathrooms

# --- Age: uniform distribution (equally likely to be any age) ---
age = rng.uniform(low=0, high=80, size=N_HOUSES)          # 0–80 years old

# --- Price: calculated from features + normally-distributed noise ---
noise = rng.normal(loc=0, scale=30000, size=N_HOUSES)     # market randomness
price = (
    140 * house_size           # $140 per sq ft
    + 20000 * bedrooms         # $20K per bedroom
    + 15000 * bathrooms        # $15K per bathroom
    - 1200 * age               # -$1200 per year of age
    + noise                    # random market variation
)
price = np.clip(price, 60000, 1500000)  # realistic price range

print(f"=== Synthetic House Price Dataset (n={N_HOUSES}) ===")
print(f"House sizes:   mean={house_size.mean():.0f} sqft, std={house_size.std():.0f} sqft")
print(f"Bedrooms:      mean={bedrooms.mean():.1f}, range=[{bedrooms.min()}, {bedrooms.max()}]")
print(f"Bathrooms:     mean={bathrooms.mean():.1f}, range=[{bathrooms.min()}, {bathrooms.max()}]")
print(f"Ages:          mean={age.mean():.1f} yrs, std={age.std():.1f} yrs")
print(f"Prices:        mean=${price.mean():,.0f}, std=${price.std():,.0f}")
print(f"Price range:   ${price.min():,.0f} – ${price.max():,.0f}")

# --- Demonstrate random sampling: bootstrap sample ---
# choice() samples WITH replacement — useful for bootstrapping
sample_indices = rng.choice(N_HOUSES, size=100, replace=False)  # 100 unique houses
sample_prices  = price[sample_indices]
print(f"\nRandom sample of 100 houses — mean price: ${sample_prices.mean():,.0f}")

# --- Train/test split using permutation ---
# permutation() gives a shuffled sequence of indices
all_indices = rng.permutation(N_HOUSES)  # shuffled [0, 1, 2, ..., 999]
split_point = int(0.8 * N_HOUSES)        # 80% train, 20% test
train_idx   = all_indices[:split_point]  # first 800 indices
test_idx    = all_indices[split_point:]  # last 200 indices

print(f"\nTrain/test split: {len(train_idx)} train, {len(test_idx)} test")
print(f"Train price mean: ${price[train_idx].mean():,.0f}")
print(f"Test price mean:  ${price[test_idx].mean():,.0f}  (should be similar — random split)")

In [ ]:
# ============================================================
# VISUALIZE THE SYNTHETIC DATASET
# Histograms of each feature + price distribution
# ============================================================
fig, axes = plt.subplots(2, 3, figsize=(16, 9))

# --- Size histogram ---
axes[0, 0].hist(house_size, bins=40, color='steelblue', edgecolor='white', alpha=0.85)
axes[0, 0].axvline(house_size.mean(), color='red', linewidth=2, linestyle='--',
                   label=f'Mean = {house_size.mean():.0f} sqft')
axes[0, 0].set_xlabel('House Size (sq ft)')
axes[0, 0].set_ylabel('Count')
axes[0, 0].set_title('House Size Distribution\n(Normal distribution, np.random.normal)')
axes[0, 0].legend(fontsize=9)

# --- Bedrooms bar chart ---
unique_beds, bed_counts = np.unique(bedrooms, return_counts=True)
axes[0, 1].bar(unique_beds, bed_counts, color='coral', edgecolor='white', alpha=0.85)
axes[0, 1].set_xlabel('Number of Bedrooms')
axes[0, 1].set_ylabel('Count')
axes[0, 1].set_title('Bedroom Count\n(Discrete, np.random.integers)')

# --- Age histogram ---
axes[0, 2].hist(age, bins=30, color='green', edgecolor='white', alpha=0.8)
axes[0, 2].set_xlabel('House Age (years)')
axes[0, 2].set_ylabel('Count')
axes[0, 2].set_title('House Age Distribution\n(Uniform, np.random.uniform)')

# --- Price histogram (raw) ---
axes[1, 0].hist(price, bins=40, color='purple', edgecolor='white', alpha=0.8)
axes[1, 0].set_xlabel('Price ($)')
axes[1, 0].set_ylabel('Count')
axes[1, 0].set_title(f'Price Distribution\nMean=${price.mean()/1000:.0f}K, Std=${price.std()/1000:.0f}K')
axes[1, 0].xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x/1000:.0f}K'))

# --- Log-price histogram (more normal) ---
log_price = np.log(price)  # log transform
axes[1, 1].hist(log_price, bins=40, color='orange', edgecolor='white', alpha=0.8)
axes[1, 1].set_xlabel('log(Price)')
axes[1, 1].set_ylabel('Count')
axes[1, 1].set_title('Log-Price Distribution\n(np.log — more symmetric, better for linear models)')

# --- Size vs Price scatter ---
axes[1, 2].scatter(house_size, price, alpha=0.15, s=10, color='steelblue')
axes[1, 2].set_xlabel('House Size (sq ft)')
axes[1, 2].set_ylabel('Price ($)')
axes[1, 2].set_title(f'Size vs Price\nr = {np.corrcoef(house_size, price)[0,1]:.3f}')
axes[1, 2].yaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f'${x/1000:.0f}K'))

plt.suptitle(f'Synthetic House Price Dataset (n={N_HOUSES:,}) — Generated with NumPy Random',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# PRACTICAL: Build and preprocess the full feature matrix
# This is the exact workflow used before feeding data to sklearn
# ============================================================
print("=== Full ML Preprocessing Pipeline with NumPy ===")

# Step 1: Stack features into a matrix (n_samples × n_features)
# np.column_stack: takes 1D arrays and stacks them as columns
X_full = np.column_stack([house_size, bedrooms, bathrooms, age])
y_full = price.copy()  # target variable

print(f"Step 1: Feature matrix shape: {X_full.shape}")
print(f"        Target shape: {y_full.shape}")

# Step 2: Check for issues
print(f"\nStep 2: Data quality checks")
print(f"  NaN values in X: {np.isnan(X_full).sum()}")
print(f"  Inf values in X: {np.isinf(X_full).sum()}")
print(f"  NaN values in y: {np.isnan(y_full).sum()}")

# Step 3: Log-transform the target (price is right-skewed)
y_log = np.log(y_full)  # log prices are more normally distributed
print(f"\nStep 3: Log-transform target")
print(f"  y (price) skewness (raw):     {((y_full - y_full.mean())**3).mean() / y_full.std()**3:.3f}")
print(f"  y (log price) skewness:       {((y_log  - y_log.mean())**3).mean()  / y_log.std()**3:.3f}")
print(f"  (Closer to 0 = more symmetric = better for linear models)")

# Step 4: Standardize features using broadcasting
X_mean = X_full.mean(axis=0)   # shape (4,)
X_std  = X_full.std(axis=0)    # shape (4,)
X_scaled = (X_full - X_mean) / X_std  # broadcasting: (1000,4) minus (4,)

print(f"\nStep 4: Standardize features (broadcasting)")
print(f"  X_scaled shape: {X_scaled.shape}")
print(f"  Column means (should be ~0): {X_scaled.mean(axis=0).round(6)}")
print(f"  Column stds  (should be ~1): {X_scaled.std(axis=0).round(4)}")

# Step 5: Train/test split using fancy indexing
rng_split = np.random.default_rng(42)
shuffled  = rng_split.permutation(N_HOUSES)  # random order of indices
n_train   = int(0.8 * N_HOUSES)

X_train, X_test = X_scaled[shuffled[:n_train]], X_scaled[shuffled[n_train:]]
y_train, y_test = y_log[shuffled[:n_train]],   y_log[shuffled[n_train:]]

print(f"\nStep 5: Train/test split")
print(f"  X_train: {X_train.shape}, y_train: {y_train.shape}")
print(f"  X_test:  {X_test.shape},  y_test:  {y_test.shape}")

print(f"\nAll steps done! Data is ready for sklearn or manual model fitting.")
print(f"  X_train and X_test are properly scaled NumPy arrays (float64)")
print(f"  y_train and y_test are log-transformed price arrays")

In [ ]:
# ============================================================
# BONUS: Useful NumPy operations for ML
# ============================================================
print("=== Dot Product: the heart of linear regression ===")
# In linear regression: predictions = X @ weights + bias
# '@' is the matrix multiplication operator in Python 3

n_features = X_train.shape[1]
# Random initial weights (how neural nets start)
weights_init = rng.normal(0, 0.1, size=n_features)
bias = 0.0

# Compute predictions for the first 5 training houses
# X_train[:5] is (5, 4), weights_init is (4,) → result is (5,)
predictions_first5 = X_train[:5] @ weights_init + bias
print(f"Predictions (5 houses): {predictions_first5.round(4)}")
print(f"Actual log prices:      {y_train[:5].round(4)}")
print(f"(These predictions are bad — weights are random; training would improve them)")

print("\n=== Vectorized Mean Squared Error (MSE) ===")
# MSE = mean of (prediction - actual)^2 — no loops needed
errors = predictions_first5 - y_train[:5]    # element-wise subtraction
squared_errors = errors ** 2                  # element-wise squaring
mse = np.mean(squared_errors)                 # mean of the squared errors
print(f"MSE (5 houses, random weights) = {mse:.4f}")

# Or in one line:
mse_oneliner = np.mean((X_train[:5] @ weights_init - y_train[:5]) ** 2)
print(f"Same result, one line:          {mse_oneliner:.4f}")

print("\n=== Sorting and argsort ===")
sample_prices = np.array([450000, 280000, 620000, 195000, 390000])
sorted_prices  = np.sort(sample_prices)           # sorted values
sort_order     = np.argsort(sample_prices)        # INDICES that would sort the array

print(f"Original prices: {sample_prices}")
print(f"Sorted prices:   {sorted_prices}")
print(f"Sort indices:    {sort_order}  ← house at index {sort_order[0]} is cheapest")

print("\n=== Stacking arrays ===")
# hstack: horizontal stack (add columns)
extra_feature = rng.normal(0, 1, (10, 1))   # 1 new feature for 10 houses
X_aug = np.hstack([X[:10], extra_feature])  # append new column
print(f"Original X shape: {X.shape}")
print(f"Augmented X shape: {X_aug.shape}  ← added 1 feature column")

# vstack: vertical stack (add rows)
new_house = np.array([[2100, 3, 2, 12]])    # 1 new house
X_extended = np.vstack([X, new_house])       # append new row
print(f"Extended X shape: {X_extended.shape}  ← added 1 house row")

## 9. Self-Check Questions

Test your understanding before moving on.

---

### Question 1
You have a 2D NumPy array `X` with shape `(1000, 5)` — 1000 houses, 5 features. How do you select all values of the 3rd feature (column index 2)?

<details>
<summary>Click for Answer</summary>

```python
X[:, 2]
```

- `X[:, 2]` means: **all rows** (`:`) from **column 2**
- This returns a 1D array of shape `(1000,)`
- Column 2 is the **3rd** column (0-indexed: 0, 1, 2, ...)

Common variations:
```python
X[5:10, 2]    # rows 5–9, column 2 (6 houses)
X[X[:,2] > 2, :]  # all rows where 3rd feature > 2 (boolean indexing)
```
</details>

---

### Question 2
`arr = np.array([100, 200, 300, 400, 500])`. What does `arr[arr > 250]` return?

<details>
<summary>Click for Answer</summary>

```python
array([300, 400, 500])
```

**Step by step:**
1. `arr > 250` creates a boolean array: `[False, False, True, True, True]`
2. `arr[boolean_mask]` selects only the `True` positions: elements at index 2, 3, 4
3. Result: `[300, 400, 500]`

The input array has 5 elements; the result has 3 (only those > 250).
</details>

---

### Question 3
Why is `np.sum(arr)` faster than Python's built-in `sum(arr)` for large arrays?

<details>
<summary>Click for Answer</summary>

**Four reasons:**

1. **Memory layout:** NumPy arrays store raw numbers in contiguous C memory. Python `sum()` must access each element through a Python object reference (slower memory access pattern).

2. **No Python overhead:** `np.sum()` is compiled C code. Python's `sum()` runs an interpreted Python loop — each iteration has interpreter overhead (opcode dispatch, reference counting, etc.).

3. **Vectorized instructions:** NumPy uses CPU SIMD (Single Instruction Multiple Data) instructions that process 4–8 numbers simultaneously in one CPU cycle.

4. **Optimized linear algebra libraries:** NumPy links against BLAS/LAPACK — highly tuned mathematical routines that exploit CPU caches efficiently.

Real-world difference: for 1M elements, NumPy is typically 10–100x faster.
</details>

---

### Question 4
You want to standardize a feature matrix `X` with shape `(1000, 5)` — subtract each column's mean and divide by its standard deviation. Write the one-liner using broadcasting.

<details>
<summary>Click for Answer</summary>

```python
X_scaled = (X - X.mean(axis=0)) / X.std(axis=0)
```

**How broadcasting works here:**
- `X.mean(axis=0)` → shape `(5,)` — one mean per column
- `X - X.mean(axis=0)` → NumPy broadcasts `(5,)` to `(1000, 5)` by repeating the row 1000 times (conceptually, without copying)
- Same for the division by `X.std(axis=0)`
- Result: `X_scaled` has shape `(1000, 5)` with mean≈0 and std≈1 per column

**Equivalent but slow loop version:**
```python
X_scaled = np.zeros_like(X)
for j in range(5):
    X_scaled[:, j] = (X[:, j] - X[:, j].mean()) / X[:, j].std()
```
The broadcasting version does the same thing in one line with no loop.
</details>

---

### Question 5 (Bonus)
What is the difference between `np.random.seed(42)` and `rng = np.random.default_rng(42)`? When would you prefer the modern API?

<details>
<summary>Click for Answer</summary>

**`np.random.seed(42)` (legacy API):**
- Sets a global random state for the entire NumPy module
- If any library or function calls `np.random` internally, it shares this state
- Can cause hard-to-debug reproducibility issues in complex code
- Still commonly seen in tutorials and notebooks

**`rng = np.random.default_rng(42)` (modern API):**
- Creates an **independent** random generator object
- Doesn't share state with global `np.random` functions
- Better statistical properties (PCG64 algorithm vs. older Mersenne Twister)
- Fully reproducible regardless of other code running around it

**Prefer modern API when:**
- Writing library code or functions others will call
- Using multiple random generators that should be independent
- In production code (more predictable behavior)

**Legacy API is fine for:** quick experiments, tutorials, small scripts where you control the whole namespace.
</details>

## 10. Quick Reference Card

```python
import numpy as np

# === CREATION ===
np.array([1, 2, 3])           # from list
np.zeros((3, 4))              # 3×4 matrix of zeros
np.ones(5)                    # [1, 1, 1, 1, 1]
np.arange(0, 10, 2)           # [0, 2, 4, 6, 8]
np.linspace(0, 1, 5)          # [0, 0.25, 0.5, 0.75, 1]
np.eye(3)                     # 3×3 identity matrix

# === ATTRIBUTES ===
arr.shape                     # tuple of dimensions
arr.dtype                     # element data type
arr.ndim                      # number of dimensions
arr.size                      # total element count

# === INDEXING ===
arr[2]                        # 3rd element
arr[-1]                       # last element
arr[1:4]                      # elements at index 1, 2, 3
X[row, col]                   # specific cell in 2D
X[:, 2]                       # entire column 2
X[3, :]                       # entire row 3
arr[arr > 0]                  # boolean indexing

# === OPERATIONS ===
arr * 2                       # scale every element
arr1 + arr2                   # element-wise add (same shape)
np.log(arr)                   # natural log (ufunc)
np.sqrt(arr)                  # square root (ufunc)
arr @ matrix                  # matrix multiplication

# === AGGREGATION ===
np.sum(arr)                   # sum all elements
np.mean(X, axis=0)            # column means
np.std(X, axis=1)             # per-row std
np.argmax(arr)                # index of max value

# === BROADCASTING / NORMALIZATION ===
(X - X.mean(axis=0)) / X.std(axis=0)    # standardize features
(X - X.min(axis=0)) / (X.max(axis=0) - X.min(axis=0))  # min-max

# === RANDOM ===
rng = np.random.default_rng(42)         # modern API, reproducible
rng.normal(300000, 50000, 1000)         # 1000 prices from normal dist
rng.integers(1, 7, 1000)               # 1000 bedroom counts (1–6)
rng.uniform(0, 80, 1000)               # 1000 house ages
rng.permutation(1000)                   # shuffled indices for train/test
```

---

*Next notebook: 09 — Pandas: DataFrames for structured data analysis*